In [ ]:
from datasets import load_from_disk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
likes = load_from_disk('dataset-parts/yambda_likes')
dislikes = load_from_disk('dataset-parts/yambda_dislikes')
listens = load_from_disk('dataset-parts/yambda_listens')

likes_table = likes.data.table
dislikes_table = dislikes.data.table
listens_table = listens.data.table


likes_df: pd.DataFrame = pd.DataFrame.from_arrow(likes_table)
dislikes_df: pd.DataFrame = pd.DataFrame.from_arrow(dislikes_table)
listens_df: pd.DataFrame = pd.DataFrame.from_arrow(listens_table)

likes_df['event_type'] = 'like'
dislikes_df['event_type'] = 'dislike'
listens_df['event_type'] = 'listen'

# yambda_df = pd.concat([likes_df, dislikes_df, listens_df], ignore_index=True)
yambda_df = pd.concat([likes_df, dislikes_df], ignore_index=True)

yambda_df = yambda_df.sort_values('timestamp')

yambda_df

In [ ]:
yambda_df_organic = yambda_df[yambda_df['is_organic'] == 1].drop(['is_organic'], axis=1)

yambda_df_organic

In [ ]:
weak_ticks = (7 * 24 * 3600) // 5
year_ticks = (365 * 24 * 3600) // 5

weak_ticks, year_ticks

In [ ]:
max_time = yambda_df_organic['timestamp'].max()

test_start = max_time - weak_ticks
train_start = test_start - year_ticks

In [ ]:
train_df = yambda_df_organic[
    (yambda_df_organic['timestamp'] >= train_start) &
    (yambda_df_organic['timestamp'] < test_start)
]

test_df = yambda_df_organic[yambda_df_organic['timestamp'] >= test_start]

Количество уникальных айтемов и пользователей

In [ ]:
train_df['item_id'].nunique(), train_df['uid'].nunique()

In [ ]:
users_activity = train_df['uid'].value_counts()

In [ ]:
users_activity.describe()

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(users_activity, bins=250)

plt.title('Распределение количества прослушиваний по пользователям')
plt.xlabel('Количество прослушиваний')
plt.ylabel('Количество пользователей (Частота)')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
sns.boxplot(x=users_activity)

In [ ]:
quantiles = [0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99, 0.999]

for quantile in quantiles:
    print(f'Listens quntile {quantile * 100}%: {users_activity.quantile(quantile)}')

In [ ]:

groups = pd.qcut(users_activity, q=[0, 0.3, 0.6, 0.9, 0.95, 1.0], duplicates='drop')

counts = groups.value_counts().sort_index()

print('Users activity distribution')
print(counts)

Можно увидеть, что активные/"нормальные" пользователи сидят между 60-ым 95-ым перцентилями, я бы даже поднял нижнюю границу до 70

In [ ]:
low_threshold = users_activity.quantile(0.7)
high_threshold = users_activity.quantile(0.95)

active_user_ids = users_activity[
    (users_activity >= low_threshold) & 
    (users_activity <= high_threshold)
].index

df_train_active = train_df[train_df['uid'].isin(active_user_ids)].reset_index(drop=True)

In [ ]:
df_train_active['uid'].value_counts().describe()

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(df_train_active['uid'].value_counts(), kde=True, bins=50)

plt.title('Распределение количества прослушиваний по пользователям')
plt.xlabel('Количество прослушиваний')
plt.ylabel('Количество пользователей (Частота)')
plt.grid(axis='y', alpha=0.3)
plt.show()

Стало получше. Мы убрали неактивных пользователей, убрали супер активных.

Может показаться странным, что за год люди слушали так мало треков, но эта статистика берется по данным с  explicit фидбеком, которого всегда меньше

In [ ]:
df_train_active

In [ ]:
df_train_active.uid.nunique(), df_train_active.item_id.nunique()

Сейчас у нас есть следующие формы взаимодействия:
* Лайк
* Дизлайк
* Прослушивание

Из них нужно сформировать разные матрицы взаимодействий: бинарные, знаковые, взвешанные

Пока что мы делали препроцессинг для лайков и дизлайков, поэтому попробуем поработать только с этими видами фидбека 

In [ ]:
counts = df_train_active.groupby(['uid', 'item_id'])['item_id'].transform('size')

duplicates_full = df_train_active[counts > 1].sort_values(by=['uid', 'item_id'])

duplicates_full.uid.nunique()

In [ ]:
df_train_active.event_type.value_counts()

Тут видна такая ситуация: у нас 1809 пользователей, из которых тольо 928 совершали несколько действий (лайк и дизлайк), при этом большинство ставит лайки трекам. Думаю я сделаю такие взаимодействия: 
* значения -1 для дизлайков и 1 для лайков
* значения 0.1 для дизлайков и 1 для лайков. 

Для iALS буду использовать только 2-ой вариант, а для EASE, KNN попробую оба

In [ ]:
train_df_sorted = train_df.sort_values(by=['uid', 'item_id', 'timestamp'])

final_states = train_df_sorted.drop_duplicates(subset=['uid', 'item_id'], keep='last').copy()

weight_mapping_sign = {
    'like': 1,
    'dislike': -1,
}

weight_mapping_weighted = {
    'like': 1,
    'dislike': 0.1,
}

final_states['interaction_sign'] = final_states['event_type'].map(weight_mapping_sign)
final_states['interaction_weighted'] = final_states['event_type'].map(weight_mapping_weighted)


user_item_sign = final_states[['uid', 'item_id', 'interaction_sign']].reset_index(drop=True)
user_item_weighted = final_states[['uid', 'item_id', 'interaction_weighted']].reset_index(drop=True)

In [ ]:
train_users_sign = set(user_item_sign['uid'])
train_items_sign = set(user_item_sign['item_id'])

test_clean = test_df[
    test_df['uid'].isin(train_users_sign) &
    test_df['item_id'].isin(train_items_sign)
]

In [ ]:
item2id = {k: v for v, k in enumerate(user_item_sign['item_id'].unique())}
user2id = {k: v for v, k in enumerate(user_item_sign['uid'].unique())}

id2item = {v: k for k, v in item2id.items()}
id2user = {v: k for k, v in user2id.items()}

In [ ]:
user_item_sign['uidx'] = user_item_sign['uid'].map(user2id)
user_item_sign['item_idx'] = user_item_sign['item_id'].map(item2id)

user_item_weighted['uidx'] = user_item_weighted['uid'].map(user2id)
user_item_weighted['item_idx'] = user_item_weighted['item_id'].map(item2id)

In [ ]:
user_item_sign

In [ ]:
user_item_weighted

In [ ]:
from scipy.sparse import csr_matrix

user_item_sign_sparse = csr_matrix(
    (
        user_item_sign['interaction_sign'],
        (user_item_sign['uidx'], user_item_sign['item_idx'])
    ),
    shape=(len(user2id), len(item2id))
)

user_item_weighted_sparse = csr_matrix(
    (
        user_item_weighted['interaction_weighted'],
        (user_item_weighted['uidx'], user_item_weighted['item_idx'])
    ),
    shape=(len(user2id), len(item2id))
)

Metrics import 

In [ ]:
from metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
    map_at_k,
    hit_rate_at_k,
)

Истинные взаимодействия, пока без разделения на лайки/дизлайки

In [ ]:
test_true = test_clean.groupby('uid')['item_id'].apply(set).to_dict()

UserKNN

In [ ]:
from models import UserKNN

In [ ]:
userknn_sign = UserKNN()
userknn_sign.fit(user_item_sign_sparse)

In [ ]:
userknn_weighted = UserKNN()
userknn_weighted.fit(user_item_weighted_sparse)

In [ ]:
recall_userknn_sign = recall_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=userknn_sign.predict
)
precision_userknn_sign = precision_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=userknn_sign.predict
)
ndcg_userknn_sign = ndcg_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=userknn_sign.predict
)
map_userknn_sign = map_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=userknn_sign.predict
)
hit_rate_userknn_sign = hit_rate_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=userknn_sign.predict
)

print('UserKNN sign NDCG@10:', ndcg_userknn_sign)
print('UserKNN sign Recall@10:', recall_userknn_sign)
print('UserKNN sign Precision@10:', precision_userknn_sign)
print('UserKNN sign MAP@10:', map_userknn_sign)
print('UserKNN sign HitRate@10:', hit_rate_userknn_sign)


In [ ]:
recall_userknn_weighted = recall_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=userknn_weighted.predict
)
precision_userknn_weighted = precision_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=userknn_weighted.predict
)
ndcg_userknn_weighted = ndcg_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=userknn_weighted.predict
)
map_userknn_weighted = map_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=userknn_weighted.predict
)
hit_rate_userknn_weighted = hit_rate_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=userknn_weighted.predict
)

print('UserKNN weighted NDCG@10:', ndcg_userknn_weighted)
print('UserKNN weighted Recall@10:', recall_userknn_weighted)
print('UserKNN weighted Precision@10:', precision_userknn_weighted)
print('UserKNN weighted MAP@10:', map_userknn_weighted)
print('UserKNN weighted HitRate@10:', hit_rate_userknn_weighted)


ItemKNN

In [ ]:
from models import ItemKNN

In [ ]:
itemknn_sign = ItemKNN()
itemknn_sign.fit(user_item_sign_sparse)

In [ ]:
itemknn_weighted = ItemKNN()
itemknn_weighted.fit(user_item_weighted_sparse)

In [ ]:
recall_itemknn_sign = recall_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=itemknn_sign.predict
)
precision_itemknn_sign = precision_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=itemknn_sign.predict
)
ndcg_itemknn_sign = ndcg_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=itemknn_sign.predict
)
map_itemknn_sign = map_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=itemknn_sign.predict
)
hit_rate_itemknn_sign = hit_rate_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=itemknn_sign.predict
)

print('ItemKNN sign NDCG@10:', ndcg_itemknn_sign)
print('ItemKNN sign Recall@10:', recall_itemknn_sign)
print('ItemKNN sign Precision@10:', precision_itemknn_sign)
print('ItemKNN sign MAP@10:', map_itemknn_sign)
print('ItemKNN sign HitRate@10:', hit_rate_itemknn_sign)


In [ ]:
recall_itemknn_weighted = recall_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=itemknn_weighted.predict
)
precision_itemknn_weighted = precision_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=itemknn_weighted.predict
)
ndcg_itemknn_weighted = ndcg_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=itemknn_weighted.predict
)
map_itemknn_weighted = map_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=itemknn_weighted.predict
)
hit_rate_itemknn_weighted = hit_rate_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=itemknn_weighted.predict
)

print('ItemKNN weighted NDCG@10:', ndcg_itemknn_weighted)
print('ItemKNN weighted Recall@10:', recall_itemknn_weighted)
print('ItemKNN weighted Precision@10:', precision_itemknn_weighted)
print('ItemKNN weighted MAP@10:', map_itemknn_weighted)
print('ItemKNN weighted HitRate@10:', hit_rate_itemknn_weighted)


ELSA


In [ ]:
from models import ELSA

In [ ]:
elsa_sign = ELSA()
elsa_sign.fit(user_item_sign_sparse)


In [ ]:
elsa_weighted = ELSA()
elsa_weighted.fit(user_item_weighted_sparse)


In [ ]:
recall_elsa_sign = recall_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=elsa_sign.predict,
)
precision_elsa_sign = precision_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=elsa_sign.predict,
)
ndcg_elsa_sign = ndcg_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=elsa_sign.predict,
)
map_elsa_sign = map_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=elsa_sign.predict,
)
hit_rate_elsa_sign = hit_rate_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=elsa_sign.predict,
)

print('ELSA sign NDCG@10:', ndcg_elsa_sign)
print('ELSA sign Recall@10:', recall_elsa_sign)
print('ELSA sign Precision@10:', precision_elsa_sign)
print('ELSA sign MAP@10:', map_elsa_sign)
print('ELSA sign HitRate@10:', hit_rate_elsa_sign)


In [ ]:
recall_elsa_weighted = recall_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=elsa_weighted.predict,
)
precision_elsa_weighted = precision_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=elsa_weighted.predict,
)
ndcg_elsa_weighted = ndcg_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=elsa_weighted.predict,
)
map_elsa_weighted = map_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=elsa_weighted.predict,
)
hit_rate_elsa_weighted = hit_rate_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=elsa_weighted.predict,
)

print('ELSA weighted NDCG@10:', ndcg_elsa_weighted)
print('ELSA weighted Recall@10:', recall_elsa_weighted)
print('ELSA weighted Precision@10:', precision_elsa_weighted)
print('ELSA weighted MAP@10:', map_elsa_weighted)
print('ELSA weighted HitRate@10:', hit_rate_elsa_weighted)


iALS


In [ ]:
from models import iALS

In [ ]:
ials_sign = iALS()
ials_sign.fit(user_item_sign_sparse)

In [ ]:
ials_weighted = iALS()
ials_weighted.fit(user_item_weighted_sparse)

In [ ]:
recall_ials_sign = recall_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=ials_sign.predict,
)
precision_ials_sign = precision_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=ials_sign.predict,
)
ndcg_ials_sign = ndcg_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=ials_sign.predict,
)
map_ials_sign = map_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=ials_sign.predict,
)
hit_rate_ials_sign = hit_rate_at_k(
    test_true,
    user_item_sign_sparse,
    user2id,
    id2item,
    recommend_fn=ials_sign.predict,
)

print('iALS sign NDCG@10:', ndcg_ials_sign)
print('iALS sign Recall@10:', recall_ials_sign)
print('iALS sign Precision@10:', precision_ials_sign)
print('iALS sign MAP@10:', map_ials_sign)
print('iALS sign HitRate@10:', hit_rate_ials_sign)


In [ ]:
recall_ials_weighted = recall_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=ials_weighted.predict,
)
precision_ials_weighted = precision_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=ials_weighted.predict,
)
ndcg_ials_weighted = ndcg_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=ials_weighted.predict,
)
map_ials_weighted = map_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=ials_weighted.predict,
)
hit_rate_ials_weighted = hit_rate_at_k(
    test_true,
    user_item_weighted_sparse,
    user2id,
    id2item,
    recommend_fn=ials_weighted.predict,
)

print('iALS weighted NDCG@10:', ndcg_ials_weighted)
print('iALS weighted Recall@10:', recall_ials_weighted)
print('iALS weighted Precision@10:', precision_ials_weighted)
print('iALS weighted MAP@10:', map_ials_weighted)
print('iALS weighted HitRate@10:', hit_rate_ials_weighted)


TopPopular, TopPersonal, TopPersonal+TopPopular

In [ ]:
from models import TopPopular, TopPersonal, TopPersonalTopPopular

top_popular_sign = TopPopular()
top_popular_sign.fit(user_item_sign_sparse)

top_popular_weighted = TopPopular()
top_popular_weighted.fit(user_item_weighted_sparse)

top_personal_sign = TopPersonal()
top_personal_sign.fit(user_item_sign_sparse)

top_personal_weighted = TopPersonal()
top_personal_weighted.fit(user_item_weighted_sparse)

hybrid_sign = TopPersonalTopPopular()
hybrid_sign.fit(user_item_sign_sparse)

hybrid_weighted = TopPersonalTopPopular()
hybrid_weighted.fit(user_item_weighted_sparse)


In [ ]:
def print_metrics_all(label, model, R, k=10):
    fn = model.predict
    print(f"=== {label} ===")
    print(f"Recall@{k}:", recall_at_k(test_true, R, user2id, id2item, k, recommend_fn=fn))
    print(f"Precision@{k}:", precision_at_k(test_true, R, user2id, id2item, k, recommend_fn=fn))
    print(f"NDCG@{k}:", ndcg_at_k(test_true, R, user2id, id2item, k, recommend_fn=fn))
    print(f"MAP@{k}:", map_at_k(test_true, R, user2id, id2item, k, recommend_fn=fn))
    print(f"HitRate@{k}:", hit_rate_at_k(test_true, R, user2id, id2item, k, recommend_fn=fn))
    print()


for name, m, R in [
    ("TopPopular sign", top_popular_sign, user_item_sign_sparse),
    ("TopPopular weighted", top_popular_weighted, user_item_weighted_sparse),
    ("TopPersonal sign", top_personal_sign, user_item_sign_sparse),
    ("TopPersonal weighted", top_personal_weighted, user_item_weighted_sparse),
    ("TopPersonal+TopPopular sign", hybrid_sign, user_item_sign_sparse),
    ("TopPersonal+TopPopular weighted", hybrid_weighted, user_item_weighted_sparse),
]:
    print_metrics_all(name, m, R)
